In [ ]:
import numpy as np
import pandas as pd

def gerar_dados_xenoverse_10_colunas(n=3000, semente=42):
    rng = np.random.default_rng(semente)

    # 1. Atributos base
    id_patrulheiro = [f"PT-{i:04d}" for i in range(1, n + 1)]

    # Nível do personagem (1 a 99)
    nivel = rng.normal(70, 20, n).clip(1, 99).round().astype(int)

    # Tempo de jogo em horas
    tempo_jogo_horas = (nivel * rng.uniform(1.5, 3.5, n) + rng.normal(0, 15, n)).clip(5, 500).round(1)

    # Medalhas PT (com outliers)
    medalhas_pt = rng.lognormal(mean=6.2, sigma=0.6, size=n)
    idx_outlier = rng.choice(n, size=15, replace=False)
    medalhas_pt[idx_outlier] *= rng.uniform(5, 12, size=15)
    medalhas_pt = medalhas_pt.round().astype(int)

    # Raças
    racas = np.array(["Saiyajin", "Terráqueo", "Majin", "Namekuseijin", "Freeza Clan"])
    raca = rng.choice(racas, size=n, p=[0.45, 0.25, 0.12, 0.08, 0.10])

    # Taxa de sucesso em missões (%)
    quest_sucesso = (50 + 0.4 * nivel + rng.normal(0, 8, n)).clip(20, 100).round(2)

    # --- NOVOS ATRIBUTOS (4 adicionais) ---
    # 7. Mentor Atual
    mentores = np.array(["Goku", "Vegeta", "Piccolo", "Beerus", "Whis", "Jiren", "Hit", "Nenhum"])
    mentor_atual = rng.choice(mentores, size=n, p=[0.25, 0.20, 0.15, 0.10, 0.08, 0.07, 0.05, 0.10])

    # 8. Ki Máximo (geralmente barras de 5 a 10 no jogo)
    ki_maximo = (3 + (nivel / 20) + rng.normal(0, 0.5, n)).clip(3, 10).round().astype(int) * 100

    # 9. Stamina Máxima (barras de vigor)
    stamina_maxima = (3 + (nivel / 25) + rng.normal(0, 0.5, n)).clip(3, 10).round().astype(int) * 100

    # 10. Status VIP / DLC Pass
    status_vip = rng.choice(["Ativo", "Inativo"], size=n, p=[0.35, 0.65])

    # Criando o DataFrame Limpo
    df_limpo = pd.DataFrame({
        "id_patrulheiro": id_patrulheiro,
        "raca": raca,
        "nivel": nivel,
        "tempo_jogo_horas": tempo_jogo_horas,
        "medalhas_pt": medalhas_pt,
        "quest_sucesso": quest_sucesso,
        "mentor_atual": mentor_atual,
        "ki_maximo": ki_maximo,
        "stamina_maxima": stamina_maxima,
        "status_vip": status_vip,
    })

    # 2. Criando a versão suja (injetando ruídos, nulos e inconsistências)
    df_sujo = df_limpo.copy()

    # Inserindo valores nulos (NaN) em várias colunas
    mask_nivel = rng.uniform(0, 1, n) < 0.05
    mask_medalhas = rng.uniform(0, 1, n) < 0.08
    mask_raca = rng.uniform(0, 1, n) < 0.04
    mask_mentor = rng.uniform(0, 1, n) < 0.06
    mask_ki = rng.uniform(0, 1, n) < 0.03

    df_sujo.loc[mask_nivel, "nivel"] = np.nan
    df_sujo.loc[mask_medalhas, "medalhas_pt"] = np.nan
    df_sujo.loc[mask_raca, "raca"] = None
    df_sujo.loc[mask_mentor, "mentor_atual"] = np.nan
    df_sujo.loc[mask_ki, "ki_maximo"] = np.nan

    # Inserindo inconsistências de texto e formatação
    erros_raca = rng.choice(n, size=50, replace=False)
    df_sujo.loc[erros_raca[:25], "raca"] = "Saiyajin "
    df_sujo.loc[erros_raca[25:], "raca"] = "terráqueo"

    erros_mentor = rng.choice(n, size=40, replace=False)
    df_sujo.loc[erros_mentor[:20], "mentor_atual"] = "goku" # Minúscula
    df_sujo.loc[erros_mentor[20:], "mentor_atual"] = "Vegeta  " # Espaços extras

    # Salvando os arquivos em CSV
    df_limpo.to_csv("xenoverse_patrulheiros_10col_limpo.csv", index=False)
    df_sujo.to_csv("xenoverse_patrulheiros_10col_sujo.csv", index=False)

    print("Bases de 10 colunas geradas com sucesso!")
    print(f"Total de registros: {n} | Total de colunas: {df_sujo.shape[1]}")
    return df_limpo, df_sujo

# Executando a função
df_limpo, df_sujo = gerar_dados_xenoverse_10_colunas()